# 02 — Face Recognition Setup
Module A3: OpenCV's built-in **LBPH recognizer** trained on a demo, consenting-subjects face dataset
(`sklearn`'s LFW subset) for "returning customer" detection. Swap in your own consented customer
photos the same way when moving to production.

> **Ethics note:** requires explicit customer opt-in, secure storage of face data, a retention limit,
> and bias testing across demographic groups — see the project report, Section 6.


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pickle
from collections import defaultdict
from datetime import datetime

import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_lfw_people

MODEL_DIR = "../app/models"
os.makedirs(MODEL_DIR, exist_ok=True)


In [ ]:
lfw = fetch_lfw_people(min_faces_per_person=25, resize=0.6)
customer_names = lfw.target_names
print(f"Loaded {len(lfw.images)} face images across {len(customer_names)} demo 'customers'")

by_id = defaultdict(list)
for img, label in zip(lfw.images, lfw.target):
    by_id[label].append(img)

train_imgs, train_labels, test_imgs, test_labels = [], [], [], []
for label, imgs in by_id.items():
    cut = max(1, int(len(imgs) * 0.8))
    train_imgs += imgs[:cut]; train_labels += [label] * cut
    test_imgs += imgs[cut:]; test_labels += [label] * (len(imgs) - cut)


In [ ]:
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.train([np.uint8(im * 255) for im in train_imgs], np.array(train_labels))
recognizer.save(f"{MODEL_DIR}/face_recognizer.yml")

face_db_meta = {"names": customer_names.tolist(), "created": str(datetime.now())}
with open(f"{MODEL_DIR}/face_db.pkl", "wb") as f:
    pickle.dump(face_db_meta, f)
print("Saved face_recognizer.yml + face_db.pkl to", MODEL_DIR)


## Quick recognition demo on a held-out 'walk-in' face

In [ ]:
idx = 0
test_face = np.uint8(test_imgs[idx] * 255)
label, confidence = recognizer.predict(test_face)
is_known = confidence < 70
print({
    "customer": customer_names[label] if is_known else "unknown",
    "status": "returning_customer" if is_known else "new_visitor",
    "confidence_distance": round(float(confidence), 2),
})
plt.imshow(test_imgs[idx], cmap="gray"); plt.axis("off"); plt.show()
